1. Setup e import universali (da eseguire subito)

In [ ]:
# ==========================================
# BLOCCO 1: SETUP E IMPORT UNIVERSALI
# ==========================================
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import trange # Fix visuale per Jupyter locale

# Scikit-Learn (Machine Learning)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize, StandardScaler

# PyTorch (Deep Learning)
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Sezione Immagini
import torchvision
import torchvision.transforms as transforms

# --- CONFIGURAZIONE DISPOSITIVO ---
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo in uso: {dispositivo}")

# --- FIX WINDOWS DATALOADER ---
# Usa SEMPRE questa funzione per creare i DataLoader all'esame
def crea_dataloaders_sicuri(train_data, val_data, test_data, batch_size=64):
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0) # num_workers=0 SALVA IL PC
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=0) if val_data else None
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader

2. Preparazione dati - Applicazione logica per immagini (tensori 3D) o per dati testuali (CSV, DataFrame)

In [ ]:
# ==========================================
# BLOCCO 2: CARICAMENTO E PREPARAZIONE DATI
# ==========================================

from torchvision.transforms import v2

TIPO_DATASET = "IMMAGINI" # Cambia in "TESTUALE" se hai un file CSV

if TIPO_DATASET == "IMMAGINI":
    print("--- Modalità Immagini Attiva ---")
    
    # 1. Definisci le Trasformazioni (Data Augmentation)
    transform_train = transforms.Compose([
        # transforms.RandomAffine(
        #    degrees=()               #Affine per eventuale traslazione e rotazione
        #    translate=()
        # ),
        transforms.RandomHorizontalFlip(),  #mirroring orizzontale, VerticalFlip per mirroring verticale
        transforms.ToTensor()  # Converte in tensore e normalizza [0,1]
    ])
    transform_test = transforms.ToTensor()  #esclusiva conversione per il test set
    
    # Data Augmentation ALTERNATIVA
    transforms = v2.Compose([
        v2.RandomAffine(degrees=(-20, 20), translate=(0.15, 0.15)), # rotazione random di un angolo tra -20° e +20°
        v2.RandomHorizontalFlip(p=0.5),                             # flip orizzontale con probabilità del 50%
        v2.ToImage(),                                               # Composizione di trasformazioni
        v2.ToDtype(torch.float32, scale=True)                       # equivalente a ToTensor e raccomandata dalla documentaizone
    ])
    tensors = v2.Compose([
        v2.ToImage(),                                               # Composizione di trasformazioni
        v2.ToDtype(torch.float32, scale=True)])


    # 2. Carica il Dataset (Es. da torchvision o da cartella)
    # Sostituisci CIFAR10 con il dataset della traccia
    dataset_completo = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    test_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    
    # 3. Split Train/Validation (es. 90/10)
    dim_train = int(0.9 * len(dataset_completo))
    dim_val = len(dataset_completo) - dim_train
    train_data, val_data = torch.utils.data.random_split(dataset_completo, [dim_train, dim_val])
    
    # 4. Estrazione Dati Piatti per Scikit-Learn
    # ATTENZIONE: Adatta le dimensioni (es. 3*32*32 = 3072 per CIFAR, 1*28*28 = 784 per MNIST)
    X_train_ml = dataset_completo.data.reshape(-1, 3*32*32).astype(np.float32) / 255.0
    y_train_ml = np.array(dataset_completo.targets)
    X_test_ml = test_data.data.reshape(-1, 3*32*32).astype(np.float32) / 255.0
    y_test_ml = np.array(test_data.targets)

elif TIPO_DATASET == "TESTUALE":
    print("--- Modalità Tabulare/Testuale Attiva ---")
    
    # 1. Carica il CSV
    # df = pd.read_csv('tuo_file.csv')
    # X = df.drop('etichetta', axis=1).values
    # y = df['etichetta'].values
    
    # [DATI FITTIZI PER ESEMPIO]
    X, y = np.random.rand(1000, 20), np.random.randint(0, 2, 1000)
    
    # 2. Split Train e Test
    X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 3. Standardizzazione (Fondamentale per le Reti Neurali su dati tabulari)
    scaler = StandardScaler()
    X_train_ml = scaler.fit_transform(X_train_ml)
    X_test_ml = scaler.transform(X_test_ml)
    
    # 4. Creazione Tensori e Dataloader per PyTorch
    train_data = TensorDataset(torch.FloatTensor(X_train_ml), torch.LongTensor(y_train_ml))
    test_data = TensorDataset(torch.FloatTensor(X_test_ml), torch.LongTensor(y_test_ml))
    val_data = None # Se non richiesto dalla traccia

3. Machine Learning - Classifier (SKLearn)

In [ ]:
# ==========================================
# BLOCCO 3: MACHINE LEARNING (GRADIENT BOOSTING / RANDOM FOREST)
# ==========================================
print("--- Avvio Training Machine Learning ---")

# Griglia snellita anti-panico (Esegue in fretta)
param_grid = {
    'learning_rate': [0.1],      # Aggiungi valori solo alla fine dell'esame
    'max_iter': [50, 100], 
    'min_samples_leaf': [20]
}

clf = HistGradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(
    clf, 
    param_grid, 
    cv=2,  # CV=2 per velocità
    n_jobs=-1, 
    verbose=1)  #verbose=3 per monitoraggio

# MODO SVILUPPO: Usa solo 1000 campioni per non bloccare il PC!
# All'ultimo minuto, scommenta la seconda riga e commenta la prima.
grid_search.fit(X_train_ml[:1000], y_train_ml[:1000]) 
# grid_search.fit(X_train_ml, y_train_ml)

best_ml = grid_search.best_estimator_
print(f"Migliori Parametri trovati: {grid_search.best_params_}")
print(f"Accuracy ML (Test): {accuracy_score(y_test_ml, best_ml.predict(X_test_ml)):.4f}")

4. Rete Neurale (PyTorch)

In [ ]:
# ==========================================
# BLOCCO 4: RETE NEURALE (PYTORCH)
# ==========================================

# SCEGLI LA RETE IN BASE AL DATASET
if TIPO_DATASET == "IMMAGINI":
    class Modello(nn.Module):
        def __init__(self):
            super().__init__()
            # Per CIFAR (3 canali, 32x32)
            self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
            self.pool = nn.MaxPool2d(2, 2)
            self.fc1 = nn.Linear(32 * 16 * 16, 128) # Adatta i calcoli in base ai pool!
            self.dropout = nn.Dropout(0.2)
            self.fc2 = nn.Linear(128, 10) # 10 classi

        def forward(self, x):
            x = self.pool(F.relu(self.conv1(x)))
            x = torch.flatten(x, 1)
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
            return x

elif TIPO_DATASET == "TESTUALE":
    class Modello(nn.Module):
        def __init__(self):
            super().__init__()
            # Rete Densa Semplice (MLP)
            self.fc1 = nn.Linear(X_train_ml.shape[1], 64) # Input = numero di feature (colonne)
            self.dropout = nn.Dropout(0.2)
            self.fc2 = nn.Linear(64, 32)
            self.fc3 = nn.Linear(32, 2) # Sostituisci 2 con il numero di classi

        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = F.relu(self.fc2(x))
            x = self.fc3(x)
            return x

# --- INIZIALIZZAZIONE E ADDESTRAMENTO ---
modello = Modello().to(dispositivo)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(modello.parameters(), lr=0.01, momentum=0.9)

train_loader, val_loader, test_loader = crea_dataloaders_sicuri(train_data, val_data, test_data)

# QUI INSERISCI IL LOOP DI ADDESTRAMENTO DEL TUO PROFESSORE
# Esempio:
# train_test(model=modello, optimizer=optimizer, device=dispositivo, 
#            train_dataloader=train_loader, test_dataloader=test_loader, epochs=5, ...)

5. Valutazione finale (ROC, AUC, Confusion Matrix)

In [ ]:
# ==========================================
# BLOCCO 5: CONFRONTO E VALUTAZIONE
# ==========================================
print("--- Valutazione Modelli ---")

# 1. Previsioni Rete Neurale
modello.eval()
y_true_nn, y_pred_nn, y_prob_nn = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader: 
        X_batch, y_batch = X_batch.to(dispositivo), y_batch.to(dispositivo)
        logits = modello(X_batch)
        probs = torch.softmax(logits, dim=1)
        
        y_true_nn.extend(y_batch.cpu().numpy())
        y_pred_nn.extend(logits.argmax(1).cpu().numpy())
        y_prob_nn.extend(probs.cpu().numpy())

y_true_nn = np.array(y_true_nn)
y_prob_nn = np.array(y_prob_nn)

# 2. Matrici di Confusione
print("\nMatrice Confusione ML:\n", confusion_matrix(y_test_ml, best_ml.predict(X_test_ml)))
print("\nMatrice Confusione PyTorch:\n", confusion_matrix(y_true_nn, y_pred_nn))

# 3. Curve ROC Multiclasse (Se la traccia lo richiede)
NUM_CLASSI = len(np.unique(y_test_ml))
y_test_bin_ml = label_binarize(y_test_ml, classes=range(NUM_CLASSI))
y_test_bin_nn = label_binarize(y_true_nn, classes=range(NUM_CLASSI))
y_prob_ml = best_ml.predict_proba(X_test_ml)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for i in range(NUM_CLASSI):
    # ML Classico
    if y_prob_ml.shape[1] > 1: # Protezione per test binari
        fpr_ml, tpr_ml, _ = roc_curve(y_test_bin_ml[:, i], y_prob_ml[:, i])
        ax1.plot(fpr_ml, tpr_ml, label=f'Class {i}')
    # Rete Neurale
    fpr_nn, tpr_nn, _ = roc_curve(y_test_bin_nn[:, i], y_prob_nn[:, i])
    ax2.plot(fpr_nn, tpr_nn, label=f'Class {i}')

ax1.set_title('ROC - ML Classico'); ax1.legend()
ax2.set_title('ROC - Rete Neurale'); ax2.legend()
plt.show()